# 2. Forecast turnout for Creative One (24 Jan 2026)

Trains a model on the Higgsfield registrations using only features that also exist in the Creative One form, checks it on a 20% validation split, then maps the bilingual (Kazakh/Russian) Creative One form to the same features and predicts how many registered people will come.

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

# ==========================================
# 1. DEFINE HELPER FUNCTIONS
# ==========================================
def group_major_bilingual(major):
    m = str(major).lower().strip()
    if any(x in m for x in ['data', 'ai', 'аналит', 'интеллект', 'intelligence']): return 'Data Science & AI'
    if any(x in m for x in ['comput', 'soft', 'cs', 'программ', 'информ', 'it']): return 'Computer Science'
    if any(x in m for x in ['cyber', 'безопасн', 'систем', 'security']): return 'IT & Cybersecurity'
    if any(x in m for x in ['engin', 'инженер', 'механ', 'строй', 'robot', 'робот']): return 'Engineering'
    if any(x in m for x in ['econ', 'finan', 'accoun', 'эконом', 'финанс', 'учет', 'аудит']): return 'Economics & Finance'
    if any(x in m for x in ['manag', 'marke', 'busin', 'менедж', 'маркет', 'бизнес', 'mba']): return 'Business & Management'
    if any(x in m for x in ['polit', 'relat', 'socio', 'psir', 'journ', 'psych', 'право', 'юрисп', 'филол', 'лингв', 'literature']): return 'Social Sciences & Humanities'
    return 'Other'

def get_reg_cat(days):
    if days <= 1: return "Less than day"
    return f"Day {int(max(0, days-1))}"

# ==========================================
# 2. TRAIN NEW NEUTRAL MODEL (Higgsfield)
# ==========================================
df_train = pd.read_csv('data/Higgsfield_Registrations.csv')  # private, not included
event_date_higgs = pd.Timestamp('2025-10-18')

# Basic cleaning
df_train = df_train.dropna(subset=['Your name and surname:', 'Your date of birth:'])
projects_list = ['AIESEC Recruitment', 'Creative One', 'GO Exchanges', 'SheCodes', 'OZGE Forum', 'EmployMe', 'Other', 'Did not participate']
df_train[projects_list] = df_train[projects_list].notna()

# Feature Engineering for Higgsfield
df_train['Major_Category'] = df_train['Your major:'].apply(group_major_bilingual)
df_train['Your place of study:'] = df_train['Your place of study:'].astype(str).str.lower().str.strip()

# Age calculation (Higgs)
df_train['Your date of birth:'] = pd.to_datetime(df_train['Your date of birth:'], errors='coerce').dt.tz_localize(None)
df_train['Age'] = (event_date_higgs - df_train['Your date of birth:']).dt.days / 365.25

# Reg Category calculation (Higgs)
df_train['Submit Date (UTC)'] = pd.to_datetime(df_train['Submit Date (UTC)']).dt.tz_localize(None)
df_train['days_before'] = (event_date_higgs - df_train['Submit Date (UTC)'].dt.normalize()).dt.days
df_train['Reg_Day_Category'] = df_train['days_before'].apply(get_reg_cat)

# SELECT NEUTRAL FEATURES
features = ['Your education degree:', 'Your place of study:', 'Major_Category', 'Reg_Day_Category', 'Age'] + projects_list
X = df_train[features]
y = df_train['CheckIn']

# Train/Test Split
cat_features = ['Your education degree:', 'Your place of study:', 'Major_Category', 'Reg_Day_Category']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit Model
neutral_model = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, cat_features=cat_features, eval_metric='AUC', verbose=False)
neutral_model.fit(X_train, y_train, eval_set=(X_test, y_test))

# --- SHOW ACCURACY ON HIGGSFIELD DATA ---
higgs_preds = neutral_model.predict(X_test)
higgs_probs = neutral_model.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, higgs_preds)
auc_score = roc_auc_score(y_test, higgs_probs)

print(f"--- HIGGSFIELD MODEL PERFORMANCE ---")
print(f"Validation Accuracy: {round(accuracy * 100, 2)}%")
print(f"Model AUC Score: {round(auc_score, 4)}")
print("------------------------------------")

# ==========================================
# 3. PREPROCESS CREATIVE ONE (Jan 24, 2026)
# ==========================================
df_creative = pd.read_csv("data/C1.csv")  # private, not included
event_date_creative = pd.Timestamp('2026-01-24')

column_mapping = {
    '*Аты-жөніңіз: / Ваше ФИО:*': 'Your name and surname:',
    '*Туған күніңіз: / Ваша дата рождения:*': 'Your date of birth:',
    '*Сіздің білім деңгейіңіз: / Ваш уровень образования:*': 'Your education degree:',
    '*Сіздің оқу орныңыз: / Ваше место обучения:*': 'Your place of study:',
    '*Сіздің мамандығыңыз: / Ваша специальность:*': 'Your major:',
    'Submitted At': 'Submit Date (UTC)',
}
df_creative = df_creative.rename(columns=column_mapping)

# Robust Mapping University & Degree
def map_uni(uni):
    uni = str(uni).lower()
    if any(x in uni for x in ['nazarbayev', 'назарбаев']): return 'nazarbayev university'
    if any(x in uni for x in ['astana it', 'aitu']): return 'astana it university'
    if any(x in uni for x in ['гумилев', 'enu', 'еуразия']): return 'eurasian national university'
    if any(x in uni for x in ['narikbayev', 'kazguu', 'қазгу']): return 'maqsut narikbayev university'
    if any(x in uni for x in ['сейфуллин', 'kazatu']): return 'kazakh agrotechnical university'
    return 'other'

def map_deg(deg):
    deg = str(deg)
    if '1 курс' in deg: return '1st Year Undergraduate Student'
    if '2 курс' in deg: return '2nd Year Undergraduate Student'
    if '3 курс' in deg: return '3rd Year Undergraduate Student'
    if '4 курс' in deg: return '4th Year Undergraduate Student'
    if 'Магистрант' in deg: return "Master's Student"
    if any(x in deg for x in ['түлегі', 'Выпускник']): return 'University Graduate'
    return 'Other'

df_creative['Your place of study:'] = df_creative['Your place of study:'].apply(map_uni)
df_creative['Your education degree:'] = df_creative['Your education degree:'].apply(map_deg)
df_creative['Major_Category'] = df_creative['Your major:'].apply(group_major_bilingual)

# Project Extraction
project_col = '*AIESEC ұйымының қандай жобаларына қатысқансыз? / В каких проектах от AIESEC Вы принимали участие ранее? *'
for p in projects_list:
    if p == 'Did not participate':
        df_creative[p] = df_creative[project_col].str.contains('Не принимал|Қатыспағанмын', na=True)
    else:
        df_creative[p] = df_creative[project_col].str.contains(p, case=False, na=False)

# Engineering (Relative to Jan 24, 2026)
df_creative['Your date of birth:'] = pd.to_datetime(df_creative['Your date of birth:'], errors='coerce')
df_creative['Age'] = (event_date_creative - df_creative['Your date of birth:']).dt.days / 365.25
df_creative['Age'] = df_creative['Age'].fillna(df_creative['Age'].mean()).round(2)

df_creative['Submit Date (UTC)'] = pd.to_datetime(df_creative['Submit Date (UTC)'], errors='coerce')
df_creative['days_before'] = (event_date_creative - df_creative['Submit Date (UTC)'].dt.tz_localize(None).dt.normalize()).dt.days
df_creative['Reg_Day_Category'] = df_creative['days_before'].apply(get_reg_cat)

# ==========================================
# 4. PREDICT FOR CREATIVE ONE
# ==========================================
X_creative = df_creative[features]
for col in cat_features:
    X_creative[col] = X_creative[col].astype(str)

preds = neutral_model.predict(X_creative)
probs = neutral_model.predict_proba(X_creative)[:, 1]

print("\n--- CREATIVE ONE PREDICTION ---")
print(f"Total Registrations: {len(df_creative)}")
print(f"Strict Prediction (Prob > 0.5): {int(sum(preds))} delegates")
print(f"Realistic Prediction (Prob > 0.3): {int(sum(probs > 0.3))} delegates")
print(f"Realistic Attendance Rate: {round((sum(probs > 0.3)/len(df_creative))*100, 2)}%")

print(f"Realistic Prediction (Prob > 0.6): {int(sum(probs > 0.6))} delegates")
print(f"Realistic Attendance Rate: {round((sum(probs > 0.6)/len(df_creative))*100, 2)}%")